In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# ==============================================================
# PURPOSE
#   Compare monochromatic vs. polychromatic propagation
#   for an IDEAL Super-Gaussian (flat-top-like) beam.
#
#   Output:
#     1) 2D intensity at z (mono / poly) in lin + log
#     2) 2D difference maps (poly - mono), incl. |diff| log
#     3) radial averaged profiles (mono vs poly)
#     4) simple quantitative metrics + "ring contrast" metric
# ==============================================================

# -----------------------------
# Physics / beam parameters
# -----------------------------
LAM0 = 800e-9          # central wavelength [m]
FWHM_NM = 10.0         # spectral FWHM [nm]  (set to your pulse bandwidth)
NLAM = 31              # number of wavelength samples (odd is nice)

Z = 4.5                # propagation distance [m]

BEAM_DIAM = 11e-3      # beam diameter [m] (used to define beam size)
SUPER_N = 40           # super-gaussian order (higher -> harder top-hat)

# -----------------------------
# Numerical grid
# -----------------------------
N = 1024
WINDOW_FACTOR = 6.0            # simulation window width = factor * BEAM_DIAM
L = WINDOW_FACTOR * BEAM_DIAM  # total side length [m]
dx = L / N
x = (np.arange(N) - N/2) * dx
X, Y = np.meshgrid(x, x)
R = np.sqrt(X**2 + Y**2)

PAD_FACTOR = 2   # 1 (off), 2, 4

# ==============================================================
# Beam: ideal super-gaussian amplitude (phase = 0)
# ==============================================================
def make_supergauss(beam_diam=BEAM_DIAM, n_order=SUPER_N):
    w0 = beam_diam / 2.0
    U = np.exp(- (R / w0)**n_order).astype(np.complex128)
    return U

# ==============================================================
# Angular Spectrum Propagation (with optional padding)
# ==============================================================
def propagate_asm(U0, z, lam, dx):
    k0 = 2*np.pi / lam
    n = U0.shape[0]
    fx = np.fft.fftfreq(n, d=dx)
    FX, FY = np.meshgrid(fx, fx)
    kx = 2*np.pi * FX
    ky = 2*np.pi * FY
    kz = np.sqrt(k0**2 - kx**2 - ky**2 + 0j)
    H = np.exp(1j * kz * z)
    return np.fft.ifft2(np.fft.fft2(U0) * H)

def propagate_asm_padded(U0, z, lam, dx, pad_factor=2):
    if pad_factor <= 1:
        return propagate_asm(U0, z, lam, dx)

    n = U0.shape[0]
    n2 = int(pad_factor * n)
    s = (n2 - n) // 2

    U = np.zeros((n2, n2), dtype=np.complex128)
    U[s:s+n, s:s+n] = U0

    k0 = 2*np.pi / lam
    fx = np.fft.fftfreq(n2, d=dx)
    FX, FY = np.meshgrid(fx, fx)
    kx = 2*np.pi * FX
    ky = 2*np.pi * FY
    kz = np.sqrt(k0**2 - kx**2 - ky**2 + 0j)
    H = np.exp(1j * kz * z)

    Uz = np.fft.ifft2(np.fft.fft2(U) * H)
    return Uz[s:s+n, s:s+n]

# ==============================================================
# Polychromatic propagation:
#   I_poly = sum_i w_i * |U(lam_i)|^2
# (camera/time integration => intensity-weighted average across spectrum)
# ==============================================================
def gaussian_spectrum(lam0, fwhm_nm, nlam):
    fwhm = fwhm_nm * 1e-9
    sigma = fwhm / (2*np.sqrt(2*np.log(2)))
    lams = np.linspace(lam0 - 3*sigma, lam0 + 3*sigma, nlam)
    w = np.exp(-0.5*((lams - lam0)/sigma)**2)
    w /= w.sum()
    return lams, w

def propagate_polychromatic_intensity(U0, z, lam0=LAM0, fwhm_nm=FWHM_NM, nlam=NLAM, pad_factor=PAD_FACTOR):
    lams, w = gaussian_spectrum(lam0, fwhm_nm, nlam)
    I_sum = np.zeros((U0.shape[0], U0.shape[1]), dtype=np.float64)

    for lam, wi in zip(lams, w):
        Uz = propagate_asm_padded(U0, z, lam, dx, pad_factor=pad_factor)
        I_sum += wi * (np.abs(Uz)**2)

    return I_sum, lams, w

# ==============================================================
# Analysis helpers: normalization, radial average, metrics
# ==============================================================
def normalize_power(I):
    s = I.sum()
    return I / s if s > 0 else I

def normalize_peak(I):
    m = I.max()
    return I / m if m > 0 else I

def radial_average(I, nbins=400):
    # radial mean in meters
    rr = R.ravel()
    ii = I.ravel()
    rmax = rr.max()
    bins = np.linspace(0, rmax, nbins+1)
    r_mid = 0.5*(bins[:-1] + bins[1:])
    I_rad = np.full(nbins, np.nan)

    for i in range(nbins):
        m = (rr >= bins[i]) & (rr < bins[i+1])
        if np.any(m):
            I_rad[i] = ii[m].mean()
    return r_mid, I_rad

def metrics_2d(I_ref, I_test, mask=None):
    if mask is None:
        mask = np.ones_like(I_ref, dtype=bool)
    a = I_ref[mask].ravel()
    b = I_test[mask].ravel()
    d = b - a
    mae = np.mean(np.abs(d))
    rmse = np.sqrt(np.mean(d**2))
    relL2 = np.linalg.norm(d) / (np.linalg.norm(a) + 1e-12)
    # corr
    a0 = a - a.mean()
    b0 = b - b.mean()
    corr = (a0 @ b0) / (np.linalg.norm(a0)*np.linalg.norm(b0) + 1e-12)
    return {"MAE": mae, "RMSE": rmse, "relL2": relL2, "Corr": corr}

def ring_contrast_metric(I, r1, r2):
    """
    Simple "ring contrast" in an annulus:
      contrast = std(I) / mean(I) within r in [r1, r2]
    Higher -> more pronounced structure/fringes.
    """
    m = (R >= r1) & (R <= r2)
    vals = I[m]
    mu = vals.mean() + 1e-12
    return vals.std() / mu

# ==============================================================
# Plot helpers
# ==============================================================
def show_2d(I, title, log=False, zoom_mm=None):
    x_mm = x * 1e3
    if zoom_mm is not None:
        m = np.abs(x_mm) <= zoom_mm
        I_show = I[np.ix_(m, m)]
        extent = [-zoom_mm, zoom_mm, -zoom_mm, zoom_mm]
    else:
        I_show = I
        extent = [x_mm.min(), x_mm.max(), x_mm.min(), x_mm.max()]

    plt.figure(figsize=(5.6,4.6))
    if log:
        plt.imshow(np.maximum(I_show, 1e-12), origin="lower", extent=extent,
                   norm=LogNorm(vmin=1e-6, vmax=1))
    else:
        plt.imshow(I_show, origin="lower", extent=extent, vmin=0, vmax=1)
    plt.xlabel("x [mm]")
    plt.ylabel("y [mm]")
    plt.title(title)
    plt.colorbar(label="norm. intensity")
    plt.tight_layout()
    plt.show()

def show_diff(I_poly, I_mono, title, zoom_mm=None):
    x_mm = x * 1e3
    if zoom_mm is not None:
        m = np.abs(x_mm) <= zoom_mm
        A = I_poly[np.ix_(m, m)]
        B = I_mono[np.ix_(m, m)]
        extent = [-zoom_mm, zoom_mm, -zoom_mm, zoom_mm]
    else:
        A = I_poly
        B = I_mono
        extent = [x_mm.min(), x_mm.max(), x_mm.min(), x_mm.max()]

    diff = A - B
    absdiff = np.abs(diff)
    vmax = np.percentile(absdiff, 99.5) + 1e-12

    fig, ax = plt.subplots(1, 2, figsize=(11,4.4))
    im0 = ax[0].imshow(diff, origin="lower", extent=extent, vmin=-vmax, vmax=vmax)
    ax[0].set_title("poly - mono (signed)")
    ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("y [mm]")
    plt.colorbar(im0, ax=ax[0], fraction=0.046)

    im1 = ax[1].imshow(np.maximum(absdiff, 1e-12), origin="lower", extent=extent,
                       norm=LogNorm(vmin=1e-6, vmax=max(absdiff.max(), 1e-6)))
    ax[1].set_title("|poly - mono| (log)")
    ax[1].set_xlabel("x [mm]"); ax[1].set_ylabel("y [mm]")
    plt.colorbar(im1, ax=ax[1], fraction=0.046)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# ==============================================================
# MAIN
# ==============================================================
if __name__ == "__main__":
    print("=== Build ideal super-gaussian input ===")
    U0 = make_supergauss()

    print("=== Propagate monochromatic ===")
    Uz_mono = propagate_asm_padded(U0, Z, LAM0, dx, pad_factor=PAD_FACTOR)
    I_mono = np.abs(Uz_mono)**2

    print("=== Propagate polychromatic (intensity-averaged) ===")
    I_poly, lams, w = propagate_polychromatic_intensity(
        U0, Z, lam0=LAM0, fwhm_nm=FWHM_NM, nlam=NLAM, pad_factor=PAD_FACTOR
    )

    # Normalize for display/compare:
    # Use power-norm to keep total energy consistent between mono and poly
    I_mono_n = normalize_power(I_mono)
    I_poly_n = normalize_power(I_poly)

    # For plotting, also show peak-normalized versions
    I_mono_p = normalize_peak(I_mono_n)
    I_poly_p = normalize_peak(I_poly_n)

    # -----------------------------
    # Plots
    # -----------------------------
    zoom_mm = 20  # change or set to None
    show_2d(I_mono_p, f"Monochromatic @ z={Z} m (lin)", log=False, zoom_mm=zoom_mm)
    show_2d(I_poly_p, f"Polychromatic (FWHM={FWHM_NM} nm) @ z={Z} m (lin)", log=False, zoom_mm=zoom_mm)

    show_2d(I_mono_p, f"Monochromatic @ z={Z} m (log)", log=True, zoom_mm=zoom_mm)
    show_2d(I_poly_p, f"Polychromatic (FWHM={FWHM_NM} nm) @ z={Z} m (log)", log=True, zoom_mm=zoom_mm)

    show_diff(I_poly_p, I_mono_p, title=f"Difference at z={Z} m (FWHM={FWHM_NM} nm)", zoom_mm=zoom_mm)

    # -----------------------------
    # Radial profiles
    # -----------------------------
    r_m, I_m_rad = radial_average(I_mono_p, nbins=500)
    r_p, I_p_rad = radial_average(I_poly_p, nbins=500)

    r_mm = r_m * 1e3
    plt.figure(figsize=(8,3.8))
    plt.plot(r_mm, I_m_rad, label="mono")
    plt.plot(r_mm, I_p_rad, label=f"poly (FWHM={FWHM_NM} nm)")
    plt.xlabel("radius r [mm]")
    plt.ylabel("peak-normalized intensity")
    plt.title("Radial average profile")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,3.8))
    plt.plot(r_mm, np.maximum(I_m_rad, 1e-12), label="mono")
    plt.plot(r_mm, np.maximum(I_p_rad, 1e-12), label=f"poly (FWHM={FWHM_NM} nm)")
    plt.yscale("log")
    plt.ylim(1e-6, 1)
    plt.xlabel("radius r [mm]")
    plt.ylabel("intensity (log)")
    plt.title("Radial average profile (log)")
    plt.grid(True, which="both", alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # -----------------------------
    # Quantify "how different" poly is from mono
    # -----------------------------
    # Compare in 2D using power-normalized intensities (fair comparison)
    # Use a beam mask to avoid far dark region dominating
    mask = I_mono_p > 1e-4  # adjustable threshold on peak-normalized mono

    m_all = metrics_2d(I_mono_n, I_poly_n, mask=None)
    m_beam = metrics_2d(I_mono_n, I_poly_n, mask=mask)

    # Simple ring contrast in an annulus (choose radii where rings appear)
    # You may need to adjust depending on your parameters.
    r1 = 4e-3   # [m]
    r2 = 8e-3   # [m]
    C_mono = ring_contrast_metric(I_mono_p, r1, r2)
    C_poly = ring_contrast_metric(I_poly_p, r1, r2)

    print("\n=== Spectrum used ===")
    print(f"lam0 = {LAM0*1e9:.1f} nm, FWHM = {FWHM_NM:.2f} nm, samples = {NLAM}")
    print(f"lambda range approx: {lams.min()*1e9:.3f} .. {lams.max()*1e9:.3f} nm")

    print("\n=== 2D metrics: poly vs mono (power-normalized) ===")


=== Build ideal super-gaussian input ===
=== Propagate monochromatic ===
=== Propagate polychromatic (intensity-averaged) ===
